In [54]:
import pandas as pd
from pathlib import Path    
from statsmodels.stats.multitest import multipletests
from scipy import stats
import numpy as np

In [55]:
result_path = Path("../results/downstream_task")
bias_types = ["less_positive_class"]
metrics = ["AUROC"]
less_bias_strengths = ["0.1"]
method_name_replacer = {"mrs-forest": "MRS",  
                        "fw-mrs-temperature": "FW-MRS",
                        "fw-mrs-temperature-svm": "FW-MRS$_{SVM}$",
                        }
data_set_replacer = {
                   # "folktables_employment": "Employment",
                    "folktables_income": "Income",
                    "breast_cancer": "Breast Cancer",
                    "hr_analytics": "HR Analytic",
                    "loan_prediction": "Loan",
                    "diabetes": "Diabetes",
                    "german_credit": "German Credit",
                    "bank_marketing": "Bank Marketing"
                    }

In [56]:
method_pairs = []
for other_method in ("fw-mrs-temperature", "fw-mrs-temperature-svm"):
    method_pairs.append(("mrs-forest", other_method))
method_pairs

[('mrs-forest', 'fw-mrs-temperature'),
 ('mrs-forest', 'fw-mrs-temperature-svm')]

In [57]:
aurocs = []
auprcs = []
dict_list = []
for dataset in data_set_replacer.keys():
    for bias_type in bias_types:
        for method in method_name_replacer.keys():
            for bias_strength in less_bias_strengths:
                json_directory = result_path / dataset / bias_type /  bias_strength/ method / "classification_results"
                auroc_file = pd.read_json(str(json_directory / "rf_auroc_list.json"))
                dict_list.append(
                    {
                        "Method": method,
                        "Data Set": dataset,
                        "AUROC": auroc_file.values,
                        "Bias Type": bias_type,
                        "Bias Strength": bias_strength
                    }
                                  )
result_df = pd.DataFrame(data=dict_list)

In [58]:
result_df.explode("AUROC")

,Method,Data Set,AUROC,Bias Type,Bias Strength
0,mrs-forest,folktables_income,[0.8359430244056351],less_positive_class,0.1
0,mrs-forest,folktables_income,[0.835011930955406],less_positive_class,0.1
0,mrs-forest,folktables_income,[0.842693099766451],less_positive_class,0.1
0,mrs-forest,folktables_income,[0.8234077738555721],less_positive_class,0.1
0,mrs-forest,folktables_income,[0.8576006758659531],less_positive_class,0.1
...,...,...,...,...,...
20,fw-mrs-temperature-svm,bank_marketing,[0.8496463903335391],less_positive_class,0.1
20,fw-mrs-temperature-svm,bank_marketing,[0.8439506641623551],less_positive_class,0.1
20,fw-mrs-temperature-svm,bank_marketing,[0.862407529207548],less_positive_class,0.1
20,fw-mrs-temperature-svm,bank_marketing,[0.7823113799252771],less_positive_class,0.1


In [59]:
def corrected_t_test(first_values, second_values, n_folds=5.0):
    differences = first_values - second_values
    mean_differences = np.mean(differences)
    std_differences = np.std(differences)
    train_size = n_folds - 1.0
    test_size = 1.0 
    correction_factor = (1.0 / len(differences)) + (test_size / train_size)
    return mean_differences / (np.sqrt(correction_factor) * std_differences)

In [60]:
p_values = []

for dataset in data_set_replacer.keys():
    for bias_type in bias_types:
            for bias_strength in less_bias_strengths:
                for first_method_name, second_metric_name in method_pairs:
                    for metric in metrics:
                        first_metrics = result_df.loc[(result_df["Method"]==first_method_name) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & (result_df["Data Set"]==dataset)][metric].values[0]
                        second_metrics = result_df.loc[(result_df["Method"]==second_metric_name) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & (result_df["Data Set"]==dataset)][metric].values[0]
                        df = len(first_metrics) - 1
                        t_statistic = corrected_t_test(np.squeeze(first_metrics), np.squeeze(second_metrics))
                        p_values.append((1.0 - stats.t.cdf(abs(t_statistic), df)) * 2.0)
corrected_p_values = multipletests(p_values, method="fdr_bh")

In [61]:
i = 0
for dataset in data_set_replacer.keys():
    for bias_type in bias_types:
            for bias_strength in less_bias_strengths:
                for first_method_name, second_metric_name in method_pairs:
                    for metric in metrics:
                        print(f"p value for {metric}, {dataset}, {bias_type}, {bias_strength}, {first_method_name},\
{second_metric_name} is: {corrected_p_values[0][i]}, {corrected_p_values[1][i]}")
                        i += 1

p value for AUROC, folktables_income, less_positive_class, 0.1, mrs-forest,fw-mrs-temperature is: False, 0.6641271431989626
p value for AUROC, folktables_income, less_positive_class, 0.1, mrs-forest,fw-mrs-temperature-svm is: False, 0.4106304480490792
p value for AUROC, breast_cancer, less_positive_class, 0.1, mrs-forest,fw-mrs-temperature is: False, 0.41987178472122444
p value for AUROC, breast_cancer, less_positive_class, 0.1, mrs-forest,fw-mrs-temperature-svm is: False, 0.4106304480490792
p value for AUROC, hr_analytics, less_positive_class, 0.1, mrs-forest,fw-mrs-temperature is: False, 0.9858832297552285
p value for AUROC, hr_analytics, less_positive_class, 0.1, mrs-forest,fw-mrs-temperature-svm is: False, 0.9858832297552285
p value for AUROC, loan_prediction, less_positive_class, 0.1, mrs-forest,fw-mrs-temperature is: False, 0.6641271431989626
p value for AUROC, loan_prediction, less_positive_class, 0.1, mrs-forest,fw-mrs-temperature-svm is: False, 0.41987178472122444
p value for 